# Tech Challenge Fase 3 - Preparacao para Modelagem
## Pipeline de Preparacao de Dados para Machine Learning

**Disciplina**: Machine Learning Engineering - Fase 3  
**Instituicao**: FIAP PosTech - AI Engineering & Machine Learning

---

## Indice
1. [Configuracao do Ambiente](#1-configuracao)
2. [Carregamento dos Dados](#2-carregamento)
3. [Remocao de Voos Cancelados/Desviados](#3-filtro)
4. [Selecao de Features](#4-features)
5. [Encoding de Variaveis Categoricas](#5-encoding)
6. [Normalizacao e Scaling](#6-scaling)
7. [Train/Test Split](#7-split)
8. [Salvamento dos Dados Preparados](#8-salvamento)

---

## 1. Configuracao do Ambiente <a id='1-configuracao'></a>

### 1.1 Importacao de Bibliotecas

In [1]:
# ============================================================================
# IMPORTACAO DE BIBLIOTECAS
# ============================================================================

# Manipulacao de dados
import pandas as pd  # Dataframes e manipulacao de dados tabulares
import numpy as np   # Operacoes matematicas e arrays
import sys
import os

# Machine Learning - Scikit-learn
from sklearn.model_selection import train_test_split  # Split treino/validacao/teste
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder  # Normalizacao e encoding

# Adicionar path do projeto para importar modulos customizados
sys.path.append('../src')

# Importar modulos do projeto (nao serao usados, mas mantem compatibilidade)
from preprocessing.missing_values import analyze_missing_values

# Configuracoes de exibicao do pandas
pd.set_option('display.max_columns', None)  # Mostrar todas as colunas
pd.set_option('display.max_rows', 100)      # Mostrar ate 100 linhas
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 casas decimais

print("Bibliotecas importadas com sucesso!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

Bibliotecas importadas com sucesso!
Pandas: 2.1.4
NumPy: 1.26.2


### 1.2 Definicao de Constantes

In [2]:
# ============================================================================
# DEFINICAO DE CONSTANTES E PARAMETROS
# ============================================================================

# Caminho dos dados processados
PROCESSED_DATA_PATH = '../data/processed/'

# IMPORTANTE: Usamos flights_enriched.parquet que contem:
#   - Missing values ja tratados (pipeline v2.0)
#   - 51 colunas (31 originais + 20 features criadas)
#   - 5.819.073 registros (apos remover 6 linhas com SCHEDULED_TIME nulo)
ENRICHED_FILE = f'{PROCESSED_DATA_PATH}flights_enriched.parquet'

# Caminho para salvar dados preparados para modelagem
MODEL_DATA_PATH = '../data/model_ready/'
os.makedirs(MODEL_DATA_PATH, exist_ok=True)

# Parametros de modelagem
DELAY_THRESHOLD = 15      # Voo e considerado atrasado se DEPARTURE_DELAY > 15 min
RANDOM_STATE = 42         # Seed para reproducibilidade dos resultados
TEST_SIZE = 0.2           # 20% dos dados para teste final
VALIDATION_SIZE = 0.1     # 10% dos dados para validacao durante treinamento
                          # Resultado: 70% treino, 10% validacao, 20% teste

print("Configuracoes definidas!")
print(f"  - Threshold de atraso: {DELAY_THRESHOLD} minutos")
print(f"  - Test size: {TEST_SIZE*100:.0f}%")
print(f"  - Validation size: {VALIDATION_SIZE*100:.0f}%")
print(f"  - Train size: {(1-TEST_SIZE-VALIDATION_SIZE)*100:.0f}%")

Configuracoes definidas!
  - Threshold de atraso: 15 minutos
  - Test size: 20%
  - Validation size: 10%
  - Train size: 70%


---
## 2. Carregamento dos Dados <a id='2-carregamento'></a>

In [3]:
# ============================================================================
# CARREGAMENTO DO DATASET ENRIQUECIDO
# ============================================================================
# Este dataset contem:
#   - Dados originais de flights.csv (5.819.079 registros)
#   - Missing values tratados (pipeline v2.0 - reducao de 97%)
#   - 20 features de engenharia criadas (temporais, geograficas, agregadas)
#   - Total: 51 colunas
# ============================================================================

print("Carregando dataset enriquecido...")
print("Isso pode levar alguns minutos...\n")

df = pd.read_parquet(ENRICHED_FILE)

print(f"Dataset carregado com sucesso!")
print(f"Dimensoes: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Carregando dataset enriquecido...
Isso pode levar alguns minutos...

Dataset carregado com sucesso!
Dimensoes: 5,819,079 linhas x 51 colunas
Memoria utilizada: 2179.55 MB


In [4]:
# Visualizar primeiras linhas
print("Primeiras 5 linhas do dataset:")
display(df.head())

Primeiras 5 linhas do dataset:


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,IS_DELAYED,DELAY_MINUTES,HOUR,PERIOD_OF_DAY,IS_WEEKEND,SEASON,IS_MORNING_RUSH,IS_EVENING_RUSH,ORIGIN_STATE,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,DEST_STATE,DEST_LATITUDE,DEST_LONGITUDE,IS_DOMESTIC,AIRLINE_DELAY_RATE,ORIGIN_DELAY_RATE,DEST_DELAY_RATE,ROUTE,ROUTE_DELAY_RATE
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5.00,2354.00,-11.00,21.00,15.00,205.00,194.00,169.00,1448,404.00,4.00,430.00,408.00,-22.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-11.00,0,Madrugada,0,Inverno,0,0,AK,61.17,-150.00,WA,47.45,-122.31,0,0.10,0.12,0.15,ANC-SEA,0.09
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10.00,2.00,-8.00,12.00,14.00,280.00,279.00,263.00,2330,737.00,4.00,750.00,741.00,-9.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-8.00,0,Madrugada,0,Inverno,0,0,CA,33.94,-118.41,FL,26.68,-80.10,0,0.16,0.20,0.21,LAX-PBI,0.12
2,2015,1,1,4,US,840,N171US,SFO,CLT,20.00,18.00,-2.00,16.00,34.00,286.00,293.00,266.00,2296,800.00,11.00,806.00,811.00,5.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-2.00,0,Madrugada,0,Inverno,0,0,CA,37.62,-122.37,NC,35.21,-80.94,0,0.14,0.20,0.13,SFO-CLT,0.09
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20.00,15.00,-5.00,15.00,30.00,285.00,281.00,258.00,2342,748.00,8.00,805.00,756.00,-9.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-5.00,0,Madrugada,0,Inverno,0,0,CA,33.94,-118.41,FL,25.79,-80.29,0,0.16,0.20,0.17,LAX-MIA,0.14
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25.00,24.00,-1.00,11.00,35.00,235.00,215.00,199.00,1448,254.00,5.00,320.00,259.00,-21.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-1.00,0,Madrugada,0,Inverno,0,0,WA,47.45,-122.31,AK,61.17,-150.00,0,0.10,0.15,0.16,SEA-ANC,0.15


---
## 3. Remocao de Voos Cancelados e Desviados <a id='3-filtro'></a>

Para modelagem de atrasos, vamos focar apenas em voos que realmente voaram.

**Nota**: O dataset `flights_enriched.parquet` foi reprocessado e agora possui:
- Missing values tratados com pipeline v2.0 (ver [03_missing_values.ipynb](03_missing_values.ipynb))
- Features de engenharia criadas (20 features novas)
- Baseado em `flights_clean.parquet` (dataset limpo e validado)
- Pronto para preparacao de modelagem

In [5]:
# ============================================================================
# FILTRAGEM: REMOVER VOOS CANCELADOS E DESVIADOS
# ============================================================================
# Para modelagem de ATRASOS, queremos apenas voos que realmente VOARAM.
# 
# Voos cancelados:
#   - Nao tem horario de partida real (DEPARTURE_TIME = NaN)
#   - Nao faz sentido prever "atraso" de algo que nao aconteceu
#   - ~89.884 voos cancelados (1.54% do dataset)
#
# Voos desviados:
#   - Chegaram em aeroporto diferente do planejado
#   - ~15.187 voos desviados (0.26% do dataset)
#   - Removemos para focar em operacao normal
#
# RESULTADO ESPERADO: ~5.714.000 voos (98.2% do dataset original)
# ============================================================================

print("Filtrando dataset para modelagem...")
print(f"  - Total de registros ANTES do filtro: {len(df):,}")

# Contar e remover voos cancelados
if 'CANCELLED' in df.columns:
    cancelled_count = df['CANCELLED'].sum()
    df = df[df['CANCELLED'] == 0]
    print(f"  - Voos cancelados removidos: {cancelled_count:,}")

# Contar e remover voos desviados
if 'DIVERTED' in df.columns:
    diverted_count = df['DIVERTED'].sum()
    df = df[df['DIVERTED'] == 0]
    print(f"  - Voos desviados removidos: {diverted_count:,}")

print(f"  - Total de registros APOS o filtro: {len(df):,}")
print(f"\nEstes {len(df):,} registros serao usados para treinamento!")

Filtrando dataset para modelagem...
  - Total de registros ANTES do filtro: 5,819,079
  - Voos cancelados removidos: 89,884
  - Voos desviados removidos: 15,187
  - Total de registros APOS o filtro: 5,714,008

Estes 5,714,008 registros serao usados para treinamento!


---
## 4. Selecao de Features <a id='4-features'></a>

### 4.1 Definir Features para Modelagem

In [6]:
# ============================================================================
# SELECAO DE FEATURES PARA MODELAGEM
# ============================================================================
# Vamos selecionar 3 tipos de features:
#
# 1. NUMERICAS (14 features):
#    - Temporais: MONTH, DAY, DAY_OF_WEEK, HOUR
#    - Operacionais: DISTANCE, SCHEDULED_TIME
#    - Taxas historicas: *_DELAY_RATE (companhia, origem, destino, rota)
#    - Geograficas: LATITUDE/LONGITUDE (origem e destino)
#
# 2. CATEGORICAS (7 features):
#    - Identificadores: AIRLINE, ORIGIN_AIRPORT, DESTINATION_AIRPORT
#    - Temporais: PERIOD_OF_DAY, SEASON
#    - Geograficas: ORIGIN_STATE, DEST_STATE
#
# 3. BINARIAS (4 features):
#    - IS_WEEKEND, IS_MORNING_RUSH, IS_EVENING_RUSH, IS_DOMESTIC
#
# TOTAL: 25 features para modelagem
#
# TARGETS (2 variaveis):
#    - IS_DELAYED: Classificacao binaria (0/1) - voo atrasou > 15min?
#    - DELAY_MINUTES: Regressao continua - quantos minutos de atraso?
# ============================================================================

# Features numericas (14)
numeric_features = [
    'MONTH',                    # 1-12
    'DAY',                      # 1-31
    'DAY_OF_WEEK',              # 1-7 (1=Segunda, 7=Domingo)
    'HOUR',                     # 0-23 (hora da partida programada)
    'DISTANCE',                 # Distancia em milhas
    'SCHEDULED_TIME',           # Tempo de voo programado (minutos)
    'AIRLINE_DELAY_RATE',       # Taxa historica de atraso da companhia
    'ORIGIN_DELAY_RATE',        # Taxa historica de atraso do aeroporto origem
    'DEST_DELAY_RATE',          # Taxa historica de atraso do aeroporto destino
    'ROUTE_DELAY_RATE',         # Taxa historica de atraso desta rota especifica
    'ORIGIN_LATITUDE',          # Latitude do aeroporto origem
    'ORIGIN_LONGITUDE',         # Longitude do aeroporto origem
    'DEST_LATITUDE',            # Latitude do aeroporto destino
    'DEST_LONGITUDE'            # Longitude do aeroporto destino
]

# Features categoricas (7)
categorical_features = [
    'AIRLINE',                  # Codigo da companhia (14 companhias)
    'ORIGIN_AIRPORT',           # Codigo IATA origem (322 aeroportos)
    'DESTINATION_AIRPORT',      # Codigo IATA destino (322 aeroportos)
    'PERIOD_OF_DAY',            # Madrugada/Manha/Tarde/Noite
    'SEASON',                   # Primavera/Verao/Outono/Inverno
    'ORIGIN_STATE',             # Estado origem (54 estados)
    'DEST_STATE'                # Estado destino (54 estados)
]

# Features binarias (4) - Ja sao 0/1, nao precisam encoding
binary_features = [
    'IS_WEEKEND',               # 1 se sabado/domingo, 0 caso contrario
    'IS_MORNING_RUSH',          # 1 se partida entre 6-9h
    'IS_EVENING_RUSH',          # 1 se partida entre 17-20h
    'IS_DOMESTIC'               # 1 se origem = destino (mesmo estado)
]

# Variaveis target
target_classification = 'IS_DELAYED'      # Binario: atraso > 15 min?
target_regression = 'DELAY_MINUTES'       # Continuo: quantos minutos?

# Filtrar apenas features que existem no dataset
numeric_features = [f for f in numeric_features if f in df.columns]
categorical_features = [f for f in categorical_features if f in df.columns]
binary_features = [f for f in binary_features if f in df.columns]

# Todas as features (X)
all_features = numeric_features + categorical_features + binary_features

print("="*80)
print("FEATURES SELECIONADAS PARA MODELAGEM")
print("="*80)
print(f"  - Numericas: {len(numeric_features)}")
print(f"  - Categoricas: {len(categorical_features)}")
print(f"  - Binarias: {len(binary_features)}")
print(f"  - TOTAL: {len(all_features)} features")
print(f"\nTARGETS:")
print(f"  - Classificacao: {target_classification}")
print(f"  - Regressao: {target_regression}")
print("="*80)

FEATURES SELECIONADAS PARA MODELAGEM
  - Numericas: 14
  - Categoricas: 7
  - Binarias: 4
  - TOTAL: 25 features

TARGETS:
  - Classificacao: IS_DELAYED
  - Regressao: DELAY_MINUTES


### 4.2 Remover Linhas com Missing Values nas Features Selecionadas

In [7]:
# ============================================================================
# VERIFICACAO DE MISSING VALUES NAS FEATURES SELECIONADAS
# ============================================================================
# Embora o dataset ja tenha passado pelo pipeline de missing values,
# algumas colunas ainda podem ter NaN propositalmente:
#   - Voos cancelados: DEPARTURE_TIME, ARRIVAL_TIME, etc (ja removidos)
#   - Features que nao foram criadas para todos os registros
#
# Aqui verificamos se as FEATURES SELECIONADAS tem missing values.
# Se tiverem, removemos essas linhas para garantir dataset limpo.
# ============================================================================

# Verificar missing values nas features selecionadas + targets
features_with_targets = all_features + [target_classification, target_regression]
missing_in_features = df[features_with_targets].isnull().sum()
missing_in_features = missing_in_features[missing_in_features > 0]

if len(missing_in_features) > 0:
    print("Features com valores ausentes:")
    for feat, count in missing_in_features.items():
        pct = count / len(df) * 100
        print(f"  - {feat}: {count:,} ({pct:.2f}%)")
    
    # Remover linhas com missing
    rows_before = len(df)
    df = df[features_with_targets].dropna()
    rows_removed = rows_before - len(df)
    
    print(f"\nLinhas removidas: {rows_removed:,} ({rows_removed/rows_before*100:.2f}%)")
else:
    print("Nenhum valor ausente nas features selecionadas!")
    df = df[features_with_targets]

print(f"\n{'='*80}")
print(f"DATASET FINAL PARA MODELAGEM: {len(df):,} linhas x {len(features_with_targets)} colunas")
print(f"  - Features (X): {len(all_features)} colunas")
print(f"  - Targets (y): 2 colunas ({target_classification}, {target_regression})")
print(f"{'='*80}")

Features com valores ausentes:
  - ORIGIN_LATITUDE: 487,439 (8.53%)
  - ORIGIN_LONGITUDE: 487,439 (8.53%)
  - DEST_LATITUDE: 487,447 (8.53%)
  - DEST_LONGITUDE: 487,447 (8.53%)
  - ORIGIN_STATE: 482,878 (8.45%)
  - DEST_STATE: 482,878 (8.45%)

Linhas removidas: 492,008 (8.61%)

DATASET FINAL PARA MODELAGEM: 5,222,000 linhas x 27 colunas
  - Features (X): 25 colunas
  - Targets (y): 2 colunas (IS_DELAYED, DELAY_MINUTES)


---
## 5. Encoding de Variaveis Categoricas <a id='5-encoding'></a>

### 5.1 Label Encoding para Features de Alta Cardinalidade

In [8]:
# ============================================================================
# ENCODING DE VARIAVEIS CATEGORICAS
# ============================================================================
# Modelos de ML nao trabalham diretamente com texto (strings).
# Precisamos converter variaveis categoricas em numeros.
#
# Estrategia usada:
#
# 1. LABEL ENCODING (alta cardinalidade - muitas categorias):
#    - ORIGIN_AIRPORT: 322 aeroportos -> 0-321
#    - DESTINATION_AIRPORT: 322 aeroportos -> 0-321
#    - ORIGIN_STATE: 54 estados -> 0-53
#    - DEST_STATE: 54 estados -> 0-53
#    
#    Por que Label? One-Hot criaria 322+322+54+54 = 752 colunas!
#    Label Encoding: 1 coluna por feature (mais eficiente)
#
# 2. ONE-HOT ENCODING (baixa cardinalidade - poucas categorias):
#    - AIRLINE: 14 companhias -> 13 colunas (drop_first=True)
#    - PERIOD_OF_DAY: 4 periodos -> 3 colunas
#    - SEASON: 4 estacoes -> 3 colunas
#    
#    Por que One-Hot? Preserva distancia entre categorias.
#    Label implicaria ordem (Verao=0, Inverno=1), o que nao faz sentido.
#
# IMPORTANTE: Salvamos os encoders para usar em producao!
# ============================================================================

# Separar features por tipo de encoding
high_cardinality_features = ['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'ORIGIN_STATE', 'DEST_STATE']
high_cardinality_features = [f for f in high_cardinality_features if f in categorical_features]

low_cardinality_features = [f for f in categorical_features if f not in high_cardinality_features]

print("="*80)
print("ENCODING DE VARIAVEIS CATEGORICAS")
print("="*80)
print(f"\n1. LABEL ENCODING (alta cardinalidade):")

df_encoded = df.copy()
label_encoders = {}

for feature in high_cardinality_features:
    le = LabelEncoder()
    df_encoded[feature] = le.fit_transform(df_encoded[feature])
    label_encoders[feature] = le
    print(f"  - {feature}: {len(le.classes_)} categorias -> valores 0-{len(le.classes_)-1}")

print(f"\n2. ONE-HOT ENCODING (baixa cardinalidade):")
for feature in low_cardinality_features:
    n_categories = df_encoded[feature].nunique()
    print(f"  - {feature}: {n_categories} categorias -> {n_categories-1} colunas (drop_first=True)")

print("="*80)

ENCODING DE VARIAVEIS CATEGORICAS

1. LABEL ENCODING (alta cardinalidade):
  - ORIGIN_AIRPORT: 319 categorias -> valores 0-318
  - DESTINATION_AIRPORT: 319 categorias -> valores 0-318
  - ORIGIN_STATE: 54 categorias -> valores 0-53
  - DEST_STATE: 54 categorias -> valores 0-53

2. ONE-HOT ENCODING (baixa cardinalidade):
  - AIRLINE: 14 categorias -> 13 colunas (drop_first=True)
  - PERIOD_OF_DAY: 4 categorias -> 3 colunas (drop_first=True)
  - SEASON: 4 categorias -> 3 colunas (drop_first=True)


### 5.2 Aplicar One-Hot Encoding

In [9]:
# ============================================================================
# APLICACAO DE ONE-HOT ENCODING
# ============================================================================
# pd.get_dummies() cria colunas binarias (0/1) para cada categoria.
# 
# Exemplo: PERIOD_OF_DAY = [Madrugada, Manha, Tarde, Noite]
# Vira 3 colunas (drop_first=True remove 1 para evitar multicolinearidade):
#   - PERIOD_OF_DAY_Manha: 1 se Manha, 0 caso contrario
#   - PERIOD_OF_DAY_Tarde: 1 se Tarde, 0 caso contrario  
#   - PERIOD_OF_DAY_Noite: 1 se Noite, 0 caso contrario
#   - Madrugada: todas 0 (categoria de referencia)
#
# IMPACTO NO NUMERO DE COLUNAS:
#   - ANTES: 25 features (14 num + 7 cat + 4 bin)
#   - DEPOIS: ~35 features (Label Encoding nao aumenta, One-Hot sim)
# ============================================================================

if len(low_cardinality_features) > 0:
    print("Aplicando One-Hot Encoding...")
    print(f"Colunas ANTES do One-Hot: {df_encoded.shape[1]}")
    
    df_encoded = pd.get_dummies(
        df_encoded, 
        columns=low_cardinality_features,
        drop_first=True  # Remove primeira categoria para evitar multicolinearidade
    )
    
    print(f"Colunas DEPOIS do One-Hot: {df_encoded.shape[1]}")
    print(f"Novas colunas criadas: {df_encoded.shape[1] - df.shape[1]}")
else:
    print("Nenhuma feature para One-Hot Encoding.")

Aplicando One-Hot Encoding...
Colunas ANTES do One-Hot: 27
Colunas DEPOIS do One-Hot: 43
Novas colunas criadas: 16


---
## 6. Normalizacao e Scaling <a id='6-scaling'></a>

### 6.1 Identificar Features Numericas para Scaling

In [10]:
# Features numericas para normalizar (excluindo target e features ja encodadas)
features_to_scale = numeric_features.copy()

print("Features para normalizacao:")
for feat in features_to_scale:
    if feat in df_encoded.columns:
        print(f"  - {feat}")

Features para normalizacao:
  - MONTH
  - DAY
  - DAY_OF_WEEK
  - HOUR
  - DISTANCE
  - SCHEDULED_TIME
  - AIRLINE_DELAY_RATE
  - ORIGIN_DELAY_RATE
  - DEST_DELAY_RATE
  - ROUTE_DELAY_RATE
  - ORIGIN_LATITUDE
  - ORIGIN_LONGITUDE
  - DEST_LATITUDE
  - DEST_LONGITUDE


### 6.2 Aplicar StandardScaler

Vamos normalizar as features numericas usando StandardScaler (z-score normalization).

In [11]:
# ============================================================================
# NORMALIZACAO COM STANDARDSCALER (Z-SCORE)
# ============================================================================
# Por que normalizar?
#   - Features tem escalas muito diferentes:
#     * DISTANCE: 31 a 4983 milhas
#     * HOUR: 0 a 23
#     * DELAY_RATE: 0.0 a 1.0
#   
#   - Modelos baseados em distancia (KNN, SVM, Redes Neurais) sao
#     sensíveis a escala. Features com valores grandes dominam.
#
# StandardScaler aplica:
#   z = (x - media) / desvio_padrao
#   
#   Resultado: media = 0, desvio padrao = 1
#   Todas as features ficam na mesma escala!
#
# IMPORTANTE:
#   - Aplicamos APENAS em features numericas
#   - Features ja encodadas (0/1) nao precisam scaling
#   - Salvamos o scaler para usar em producao (mesma transformacao)
# ============================================================================

print("Aplicando StandardScaler nas features numericas...")
print(f"Features numericas a normalizar: {len(numeric_features)}\n")

# Criar copia para scaling
df_scaled = df_encoded.copy()

# Aplicar scaler apenas nas features numericas que ainda existem
scaler = StandardScaler()
features_existing = [f for f in numeric_features if f in df_scaled.columns]

if len(features_existing) > 0:
    # fit_transform: calcula media/std E aplica transformacao
    df_scaled[features_existing] = scaler.fit_transform(df_scaled[features_existing])
    
    print(f"StandardScaler aplicado em {len(features_existing)} features.")
    print(f"\nEstatisticas APOS scaling (devem estar proximas de 0 e 1):")
    print(df_scaled[features_existing].describe().loc[['mean', 'std']].round(4))
else:
    print("Nenhuma feature numerica para scaling.")

print("\n" + "="*80)
print(f"DATASET NORMALIZADO: {df_scaled.shape[0]:,} linhas x {df_scaled.shape[1]} colunas")
print("="*80)

Aplicando StandardScaler nas features numericas...
Features numericas a normalizar: 14

StandardScaler aplicado em 14 features.

Estatisticas APOS scaling (devem estar proximas de 0 e 1):
      MONTH  DAY  DAY_OF_WEEK  HOUR  DISTANCE  SCHEDULED_TIME  \
mean   0.00 0.00         0.00  0.00      0.00            0.00   
std    1.00 1.00         1.00  1.00      1.00            1.00   

      AIRLINE_DELAY_RATE  ORIGIN_DELAY_RATE  DEST_DELAY_RATE  \
mean               -0.00              -0.00             0.00   
std                 1.00               1.00             1.00   

      ROUTE_DELAY_RATE  ORIGIN_LATITUDE  ORIGIN_LONGITUDE  DEST_LATITUDE  \
mean              0.00             0.00             -0.00          -0.00   
std               1.00             1.00              1.00           1.00   

      DEST_LONGITUDE  
mean            0.00  
std             1.00  

DATASET NORMALIZADO: 5,222,000 linhas x 43 colunas


---
## 7. Train/Test Split <a id='7-split'></a>

### 7.1 Separar Features e Targets

In [12]:
# ============================================================================
# SEPARACAO DE FEATURES (X) E TARGETS (y)
# ============================================================================
# Em Machine Learning:
#   X = Features (variaveis independentes) - o que o modelo USA para prever
#   y = Target (variavel dependente) - o que o modelo TENTA prever
#
# Temos 2 problemas de ML neste projeto:
#
# 1. CLASSIFICACAO BINARIA:
#    - X: Todas as features (hora, distancia, companhia, etc)
#    - y: IS_DELAYED (0 ou 1)
#    - Pergunta: "Este voo vai atrasar?"
#    - Modelos: Logistic Regression, Random Forest, XGBoost
#
# 2. REGRESSAO:
#    - X: Mesmas features
#    - y: DELAY_MINUTES (valor continuo)
#    - Pergunta: "Quantos minutos de atraso?"
#    - Modelos: Linear Regression, Random Forest Regressor, XGBoost
#
# DIMENSOES ESPERADAS (apos filtros):
#   - X: ~5.714.000 linhas x ~35 colunas (varia com One-Hot)
#   - y_classification: ~5.714.000 valores (0 ou 1)
#   - y_regression: ~5.714.000 valores (numeros reais)
# ============================================================================

print("="*80)
print("SEPARACAO DE FEATURES (X) E TARGETS (y)")
print("="*80)

# Remover colunas target do dataset para obter apenas features (X)
X = df_scaled.drop(columns=[target_classification, target_regression])

# Extrair cada target separadamente
y_classification = df_scaled[target_classification]  # IS_DELAYED (0/1)
y_regression = df_scaled[target_regression]          # DELAY_MINUTES (continuo)

print(f"\nDIMENSOES:")
print(f"  - X (features):        {X.shape[0]:,} linhas x {X.shape[1]} colunas")
print(f"  - y_classification:    {y_classification.shape[0]:,} valores")
print(f"  - y_regression:        {y_regression.shape[0]:,} valores")

print(f"\nDISTRIBUICAO DE CLASSES (Classificacao):")
counts = y_classification.value_counts().sort_index()
for class_val, count in counts.items():
    pct = count / len(y_classification) * 100
    label = "NAO ATRASADO" if class_val == 0 else "ATRASADO"
    print(f"  - Classe {class_val} ({label}): {count:,} ({pct:.2f}%)")

print(f"\nESTATISTICAS DO TARGET DE REGRESSAO:")
print(f"  - Media de atraso: {y_regression.mean():.2f} minutos")
print(f"  - Mediana: {y_regression.median():.2f} minutos")
print(f"  - Min: {y_regression.min():.2f} minutos")
print(f"  - Max: {y_regression.max():.2f} minutos")
print("="*80)

SEPARACAO DE FEATURES (X) E TARGETS (y)

DIMENSOES:
  - X (features):        5,222,000 linhas x 41 colunas
  - y_classification:    5,222,000 valores
  - y_regression:        5,222,000 valores

DISTRIBUICAO DE CLASSES (Classificacao):
  - Classe 0 (NAO ATRASADO): 4,269,294 (81.76%)
  - Classe 1 (ATRASADO): 952,706 (18.24%)

ESTATISTICAS DO TARGET DE REGRESSAO:
  - Media de atraso: 9.70 minutos
  - Mediana: -1.00 minutos
  - Min: -82.00 minutos
  - Max: 1988.00 minutos


### 7.2 Train/Validation/Test Split

Vamos dividir os dados em:
- **Train**: 70% para treinamento
- **Validation**: 10% para validacao durante treinamento
- **Test**: 20% para avaliacao final

In [13]:
# ============================================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================================
# DIVISAO DOS DADOS EM 3 CONJUNTOS:
#
# 1. TREINO (70%):
#    - Usado para TREINAR o modelo (fit)
#    - Modelo aprende padroes desses dados
#    - ~4.000.000 de registros
#
# 2. VALIDACAO (10%):
#    - Usado DURANTE o treinamento para:
#      * Ajustar hiperparametros
#      * Early stopping (parar treinamento antes de overfit)
#      * Selecionar melhor modelo entre variacoes
#    - ~571.000 registros
#
# 3. TESTE (20%):
#    - NUNCA visto pelo modelo durante treinamento
#    - Usado APENAS no final para avaliacao real
#    - Simula dados novos em producao
#    - ~1.143.000 registros
#
# POR QUE STRATIFY?
#   - Dataset desbalanceado: ~82% nao atrasados, ~18% atrasados
#   - stratify=y_classification garante mesma proporcao em cada split
#   - Evita treino com 90% atrasados e teste com 10% (seria desastre!)
#
# PROCESSO DE SPLIT (2 ETAPAS):
#   1. Split 80/20: (Treino+Val) vs Teste
#   2. Split do conjunto temp: Treino (87.5%) vs Val (12.5%)
#      87.5% de 80% = 70% do total
#      12.5% de 80% = 10% do total
#
# RANDOM_STATE = 42:
#   - Garante reproducibilidade
#   - Sempre os mesmos registros em cada conjunto
#   - Permite comparar experimentos de forma justa
#
# DIMENSOES FINAIS ESPERADAS (assumindo ~5.714.000 registros):
#   X_train:  (4.000.000, 35)  y_class_train:  (4.000.000,)
#   X_val:    (571.000, 35)    y_class_val:    (571.000,)
#   X_test:   (1.143.000, 35)  y_class_test:   (1.143.000,)
#   
#   (e mesmas dimensoes para y_reg_*)
# ============================================================================

print("="*80)
print("DIVISAO EM CONJUNTOS DE TREINO / VALIDACAO / TESTE")
print("="*80)
print(f"\nTotal de registros: {len(X):,}")
print(f"Total de features: {X.shape[1]}")
print(f"\nProporcao TARGET:")
print(f"  - Classe 0 (nao atrasado): {(y_classification==0).sum():,} ({(y_classification==0).mean()*100:.2f}%)")
print(f"  - Classe 1 (atrasado):     {(y_classification==1).sum():,} ({(y_classification==1).mean()*100:.2f}%)")

# ========================================
# ETAPA 1: Split Treino+Val (80%) vs Teste (20%)
# ========================================
print(f"\nETAPA 1: Separando Teste (20%)...")
X_temp, X_test, y_class_temp, y_class_test, y_reg_temp, y_reg_test = train_test_split(
    X, y_classification, y_regression,
    test_size=TEST_SIZE,          # 20% para teste
    random_state=RANDOM_STATE,    # Seed para reproducibilidade
    stratify=y_classification     # Manter proporcao 82/18
)

print(f"  - Conjunto temporario (treino+val): {X_temp.shape[0]:,} registros")
print(f"  - Conjunto de TESTE: {X_test.shape[0]:,} registros ({len(X_test)/len(X)*100:.1f}%)")

# ========================================
# ETAPA 2: Split Treino (70%) vs Validacao (10%)
# ========================================
print(f"\nETAPA 2: Separando Validacao (10% do total)...")

# Calcular tamanho da validacao RELATIVO ao conjunto temp
# Queremos 10% do total = 10%/80% = 12.5% do temp
val_size_adjusted = VALIDATION_SIZE / (1 - TEST_SIZE)
print(f"  - Tamanho validacao ajustado: {val_size_adjusted*100:.1f}% do conjunto temp")

X_train, X_val, y_class_train, y_class_val, y_reg_train, y_reg_val = train_test_split(
    X_temp, y_class_temp, y_reg_temp,
    test_size=val_size_adjusted,
    random_state=RANDOM_STATE,
    stratify=y_class_temp
)

print(f"  - Conjunto de TREINO: {X_train.shape[0]:,} registros ({len(X_train)/len(X)*100:.1f}%)")
print(f"  - Conjunto de VALIDACAO: {X_val.shape[0]:,} registros ({len(X_val)/len(X)*100:.1f}%)")

# ========================================
# RESUMO FINAL
# ========================================
print(f"\n" + "="*80)
print("DIMENSOES FINAIS DOS CONJUNTOS")
print("="*80)
print(f"\nFEATURES (X):")
print(f"  - X_train:      {X_train.shape}  = ({X_train.shape[0]:,} registros, {X_train.shape[1]} features)")
print(f"  - X_val:        {X_val.shape}  = ({X_val.shape[0]:,} registros, {X_val.shape[1]} features)")
print(f"  - X_test:       {X_test.shape}  = ({X_test.shape[0]:,} registros, {X_test.shape[1]} features)")

print(f"\nTARGET CLASSIFICACAO (y_class):")
print(f"  - y_class_train: {y_class_train.shape}  = {y_class_train.shape[0]:,} valores")
print(f"  - y_class_val:   {y_class_val.shape}  = {y_class_val.shape[0]:,} valores")
print(f"  - y_class_test:  {y_class_test.shape}  = {y_class_test.shape[0]:,} valores")

print(f"\nTARGET REGRESSAO (y_reg):")
print(f"  - y_reg_train:   {y_reg_train.shape}  = {y_reg_train.shape[0]:,} valores")
print(f"  - y_reg_val:     {y_reg_val.shape}  = {y_reg_val.shape[0]:,} valores")
print(f"  - y_reg_test:    {y_reg_test.shape}  = {y_reg_test.shape[0]:,} valores")

print(f"\nBALANCEAMENTO DE CLASSES (% de voos atrasados):")
print(f"  - Train:      {y_class_train.mean()*100:.2f}%")
print(f"  - Validation: {y_class_val.mean()*100:.2f}%")
print(f"  - Test:       {y_class_test.mean()*100:.2f}%")
print(f"\n  Proporcoes identicas = STRATIFY funcionou!")
print("="*80)

DIVISAO EM CONJUNTOS DE TREINO / VALIDACAO / TESTE

Total de registros: 5,222,000
Total de features: 41

Proporcao TARGET:
  - Classe 0 (nao atrasado): 4,269,294 (81.76%)
  - Classe 1 (atrasado):     952,706 (18.24%)

ETAPA 1: Separando Teste (20%)...
  - Conjunto temporario (treino+val): 4,177,600 registros
  - Conjunto de TESTE: 1,044,400 registros (20.0%)

ETAPA 2: Separando Validacao (10% do total)...
  - Tamanho validacao ajustado: 12.5% do conjunto temp
  - Conjunto de TREINO: 3,655,400 registros (70.0%)
  - Conjunto de VALIDACAO: 522,200 registros (10.0%)

DIMENSOES FINAIS DOS CONJUNTOS

FEATURES (X):
  - X_train:      (3655400, 41)  = (3,655,400 registros, 41 features)
  - X_val:        (522200, 41)  = (522,200 registros, 41 features)
  - X_test:       (1044400, 41)  = (1,044,400 registros, 41 features)

TARGET CLASSIFICACAO (y_class):
  - y_class_train: (3655400,)  = 3,655,400 valores
  - y_class_val:   (522200,)  = 522,200 valores
  - y_class_test:  (1044400,)  = 1,044,400 va

---
## 8. Salvamento dos Dados Preparados <a id='8-salvamento'></a>

### 8.1 Salvar Conjuntos de Dados

In [14]:
# ============================================================================
# SALVAMENTO DOS CONJUNTOS DE DADOS PREPARADOS
# ============================================================================
# Salvamos os 3 conjuntos (train/val/test) em arquivos separados:
#
# Formato Parquet:
#   - Formato binario colunar (muito mais eficiente que CSV)
#   - ~88% menor que CSV
#   - Preserva tipos de dados (int, float, etc)
#   - Carregamento muito mais rapido
#
# Estrutura dos arquivos salvos:
#   - Colunas 0 a N-1: Features (X) - normalizadas e encodadas
#   - Coluna N: IS_DELAYED - target classificacao (0/1)
#   - Coluna N+1: DELAY_MINUTES - target regressao (float)
#
# Uso posterior:
#   train_data = pd.read_parquet('train_data.parquet')
#   X_train = train_data.drop(columns=['IS_DELAYED', 'DELAY_MINUTES'])
#   y_train = train_data['IS_DELAYED']  # ou 'DELAY_MINUTES'
# ============================================================================

print("="*80)
print("SALVANDO CONJUNTOS DE DADOS PREPARADOS")
print("="*80)

# Reconstruir DataFrames completos (features + targets)
train_data = pd.DataFrame(X_train)
train_data['IS_DELAYED'] = y_class_train.values
train_data['DELAY_MINUTES'] = y_reg_train.values

val_data = pd.DataFrame(X_val)
val_data['IS_DELAYED'] = y_class_val.values
val_data['DELAY_MINUTES'] = y_reg_val.values

test_data = pd.DataFrame(X_test)
test_data['IS_DELAYED'] = y_class_test.values
test_data['DELAY_MINUTES'] = y_reg_test.values

print(f"\nDataFrames criados:")
print(f"  - train_data: {train_data.shape}")
print(f"  - val_data:   {val_data.shape}")
print(f"  - test_data:  {test_data.shape}")

# Definir caminhos dos arquivos
train_file = f'{MODEL_DATA_PATH}train_data.parquet'
val_file = f'{MODEL_DATA_PATH}val_data.parquet'
test_file = f'{MODEL_DATA_PATH}test_data.parquet'

print(f"\nSalvando em formato Parquet...")

# Salvar
train_data.to_parquet(train_file, index=False)
val_data.to_parquet(val_file, index=False)
test_data.to_parquet(test_file, index=False)

print(f"\nArquivos salvos:")
print(f"  - Train: {train_file}")
print(f"           Tamanho: {os.path.getsize(train_file) / 1024**2:.2f} MB")
print(f"  - Val:   {val_file}")
print(f"           Tamanho: {os.path.getsize(val_file) / 1024**2:.2f} MB")
print(f"  - Test:  {test_file}")
print(f"           Tamanho: {os.path.getsize(test_file) / 1024**2:.2f} MB")
print("="*80)

SALVANDO CONJUNTOS DE DADOS PREPARADOS

DataFrames criados:
  - train_data: (3655400, 43)
  - val_data:   (522200, 43)
  - test_data:  (1044400, 43)

Salvando em formato Parquet...

Arquivos salvos:
  - Train: ../data/model_ready/train_data.parquet
           Tamanho: 73.33 MB
  - Val:   ../data/model_ready/val_data.parquet
           Tamanho: 10.52 MB
  - Test:  ../data/model_ready/test_data.parquet
           Tamanho: 20.95 MB


### 8.2 Salvar Feature Names e Metadata

In [15]:
# ============================================================================
# SALVAMENTO DE METADATA
# ============================================================================
# Salvamos informacoes importantes sobre o preprocessing:
#
# 1. feature_names: Lista de todas as features apos encoding/scaling
#    - Importante: ordem das colunas deve ser mantida em producao!
#
# 2. Configuracoes usadas:
#    - delay_threshold: 15 minutos
#    - random_state: 42
#
# 3. Tamanhos dos conjuntos:
#    - Para validar que carregamos os arquivos corretos
#
# 4. Balanceamento de classes:
#    - Para monitorar drift em producao
#    - Se producao tiver 50% atrasados, algo esta errado!
#
# Este arquivo e ESSENCIAL para colocar modelo em producao!
# ============================================================================

import json

# Obter nomes das features (apos One-Hot, podem ter nomes como 'SEASON_Verao')
feature_names = X.columns.tolist()

# Criar dicionario com todos os metadados
metadata = {
    'dataset_info': {
        'total_registros_originais': len(X),
        'n_features_apos_encoding': len(feature_names),
        'n_features_originais': len(all_features)
    },
    'features': {
        'feature_names': feature_names,
        'numeric_features': [f for f in numeric_features if f in feature_names],
        'categorical_features_encoded': high_cardinality_features,
        'binary_features': binary_features
    },
    'targets': {
        'classification': target_classification,
        'regression': target_regression,
        'delay_threshold_minutes': DELAY_THRESHOLD
    },
    'splits': {
        'train_size': len(X_train),
        'val_size': len(X_val),
        'test_size': len(X_test),
        'train_pct': len(X_train)/len(X)*100,
        'val_pct': len(X_val)/len(X)*100,
        'test_pct': len(X_test)/len(X)*100
    },
    'class_balance': {
        'train_delayed_pct': float(y_class_train.mean()),
        'val_delayed_pct': float(y_class_val.mean()),
        'test_delayed_pct': float(y_class_test.mean())
    },
    'config': {
        'random_state': RANDOM_STATE,
        'test_size': TEST_SIZE,
        'validation_size': VALIDATION_SIZE
    }
}

# Salvar em JSON (formato legivel)
metadata_file = f'{MODEL_DATA_PATH}metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print("="*80)
print("METADATA SALVO")
print("="*80)
print(f"\nArquivo: {metadata_file}")
print(f"\nConteudo:")
print(f"  - Total de features: {metadata['dataset_info']['n_features_apos_encoding']}")
print(f"  - Tamanho treino: {metadata['splits']['train_size']:,} ({metadata['splits']['train_pct']:.1f}%)")
print(f"  - Tamanho validacao: {metadata['splits']['val_size']:,} ({metadata['splits']['val_pct']:.1f}%)")
print(f"  - Tamanho teste: {metadata['splits']['test_size']:,} ({metadata['splits']['test_pct']:.1f}%)")
print(f"  - % voos atrasados (treino): {metadata['class_balance']['train_delayed_pct']*100:.2f}%")
print("="*80)

METADATA SALVO

Arquivo: ../data/model_ready/metadata.json

Conteudo:
  - Total de features: 41
  - Tamanho treino: 3,655,400 (70.0%)
  - Tamanho validacao: 522,200 (10.0%)
  - Tamanho teste: 1,044,400 (20.0%)
  - % voos atrasados (treino): 18.24%


### 8.3 Salvar Objetos de Preprocessamento

In [16]:
# ============================================================================
# SALVAMENTO DE OBJETOS DE PREPROCESSING
# ============================================================================
# CRITICO PARA PRODUCAO!
#
# Quando o modelo for para producao, NOVOS dados precisam passar pelo
# MESMO preprocessing que os dados de treino:
#
# 1. scaler (StandardScaler):
#    - Contem media e desvio padrao calculados no TREINO
#    - Usa scaler.transform(X_novo) - NAO fit_transform!
#    - Garante mesma escala que treino
#
# 2. label_encoders (dict de LabelEncoders):
#    - ORIGIN_AIRPORT: LAX -> 123, JFK -> 45, etc
#    - Mesmos mapeamentos usados no treino
#    - Se aparecer aeroporto novo, precisa tratamento especial
#
# 3. feature_names:
#    - Ordem EXATA das colunas
#    - Modelo espera features nessa ordem!
#
# Exemplo de uso em producao:
#   import pickle
#   with open('preprocessing_objects.pkl', 'rb') as f:
#       prep = pickle.load(f)
#   
#   # Aplicar encoding
#   X_novo['ORIGIN_AIRPORT'] = prep['label_encoders']['ORIGIN_AIRPORT'].transform(...)
#   
#   # Aplicar scaling
#   X_novo_scaled = prep['scaler'].transform(X_novo)
#   
#   # Garantir ordem correta
#   X_novo_scaled = X_novo_scaled[prep['feature_names']]
#   
#   # Agora pode prever!
#   y_pred = modelo.predict(X_novo_scaled)
# ============================================================================

import pickle

# Empacotar todos os objetos necessarios
preprocessing_objects = {
    'scaler': scaler,                      # StandardScaler fitted
    'label_encoders': label_encoders,      # Dict de LabelEncoders fitted
    'feature_names': feature_names         # Lista ordenada de features
}

# Salvar em formato pickle (binario)
preprocessing_file = f'{MODEL_DATA_PATH}preprocessing_objects.pkl'
with open(preprocessing_file, 'wb') as f:
    pickle.dump(preprocessing_objects, f)

print("="*80)
print("OBJETOS DE PREPROCESSING SALVOS")
print("="*80)
print(f"\nArquivo: {preprocessing_file}")
print(f"Tamanho: {os.path.getsize(preprocessing_file) / 1024:.2f} KB")
print(f"\nConteudo:")
print(f"  - scaler: StandardScaler com {len(scaler.mean_)} features")
print(f"  - label_encoders: {len(label_encoders)} encoders")
for feat, encoder in label_encoders.items():
    print(f"      * {feat}: {len(encoder.classes_)} classes")
print(f"  - feature_names: lista com {len(feature_names)} features")
print(f"\nESSE ARQUIVO E ESSENCIAL PARA COLOCAR MODELO EM PRODUCAO!")
print("="*80)

OBJETOS DE PREPROCESSING SALVOS

Arquivo: ../data/model_ready/preprocessing_objects.pkl
Tamanho: 6.11 KB

Conteudo:
  - scaler: StandardScaler com 14 features
  - label_encoders: 4 encoders
      * ORIGIN_AIRPORT: 319 classes
      * DESTINATION_AIRPORT: 319 classes
      * ORIGIN_STATE: 54 classes
      * DEST_STATE: 54 classes
  - feature_names: lista com 41 features

ESSE ARQUIVO E ESSENCIAL PARA COLOCAR MODELO EM PRODUCAO!


---

## Conclusao

Neste notebook, preparamos os dados para modelagem:

**Etapas Realizadas**:
1. Carregamento do dataset enriquecido
2. Tratamento de valores ausentes
3. Filtragem de voos cancelados/desviados
4. Selecao de features relevantes
5. Encoding de variaveis categoricas
6. Normalizacao de features numericas
7. Split em treino/validacao/teste
8. Salvamento dos dados preparados

**Proximos Passos**:
1. Modelagem de Classificacao (IS_DELAYED)
2. Modelagem de Regressao (DELAY_MINUTES)
3. Avaliacao e comparacao de modelos
4. Feature importance e interpretacao

In [17]:
# ============================================================================
# RESUMO FINAL - DADOS PRONTOS PARA MODELAGEM
# ============================================================================

print("="*80)
print("PREPARACAO COMPLETA - DADOS PRONTOS PARA MODELAGEM!")
print("="*80)

print(f"\nDIRETORIO: {MODEL_DATA_PATH}")

print(f"\nARQUIVOS GERADOS:")
print(f"  1. train_data.parquet     - {X_train.shape[0]:,} x {train_data.shape[1]} (features + 2 targets)")
print(f"  2. val_data.parquet       - {X_val.shape[0]:,} x {val_data.shape[1]} (features + 2 targets)")
print(f"  3. test_data.parquet      - {X_test.shape[0]:,} x {test_data.shape[1]} (features + 2 targets)")
print(f"  4. metadata.json          - Informacoes sobre preprocessing")
print(f"  5. preprocessing_objects.pkl - Scaler + Encoders para producao")

print(f"\nDIMENSOES FINAIS PARA TREINAMENTO:")
print(f"  ┌─────────────┬──────────────────┬──────────────────┐")
print(f"  │ Conjunto    │ Registros        │ Features         │")
print(f"  ├─────────────┼──────────────────┼──────────────────┤")
print(f"  │ TREINO      │ {X_train.shape[0]:>15,} │ {X_train.shape[1]:>16} │")
print(f"  │ VALIDACAO   │ {X_val.shape[0]:>15,} │ {X_val.shape[1]:>16} │")
print(f"  │ TESTE       │ {X_test.shape[0]:>15,} │ {X_test.shape[1]:>16} │")
print(f"  └─────────────┴──────────────────┴──────────────────┘")

print(f"\nCARACTERISTICAS DOS DADOS:")
print(f"  - Features normalizadas (StandardScaler: media=0, std=1)")
print(f"  - Variaveis categoricas encodadas (Label + One-Hot)")
print(f"  - Balanceamento preservado (stratify):")
print(f"      * Treino: {y_class_train.mean()*100:.2f}% atrasados")
print(f"      * Val:    {y_class_val.mean()*100:.2f}% atrasados")
print(f"      * Teste:  {y_class_test.mean()*100:.2f}% atrasados")

print(f"\nPROXIMOS PASSOS:")
print(f"  1. Modelagem de CLASSIFICACAO (IS_DELAYED):")
print(f"     - Logistic Regression (baseline)")
print(f"     - Random Forest Classifier")
print(f"     - XGBoost Classifier")
print(f"     - Metricas: ROC-AUC, F1-Score, Precision, Recall")
print(f"")
print(f"  2. Modelagem de REGRESSAO (DELAY_MINUTES):")
print(f"     - Linear Regression (baseline)")
print(f"     - Random Forest Regressor")
print(f"     - XGBoost Regressor")
print(f"     - Metricas: R², RMSE, MAE")
print(f"")
print(f"  3. Feature Importance e Interpretacao")
print(f"  4. Tuning de Hiperparametros")
print(f"  5. Avaliacao Final no conjunto de TESTE")

print("\n" + "="*80)
print("SUCESSO! DADOS 100% PRONTOS PARA MACHINE LEARNING")
print("="*80)

PREPARACAO COMPLETA - DADOS PRONTOS PARA MODELAGEM!

DIRETORIO: ../data/model_ready/

ARQUIVOS GERADOS:
  1. train_data.parquet     - 3,655,400 x 43 (features + 2 targets)
  2. val_data.parquet       - 522,200 x 43 (features + 2 targets)
  3. test_data.parquet      - 1,044,400 x 43 (features + 2 targets)
  4. metadata.json          - Informacoes sobre preprocessing
  5. preprocessing_objects.pkl - Scaler + Encoders para producao

DIMENSOES FINAIS PARA TREINAMENTO:
  ┌─────────────┬──────────────────┬──────────────────┐
  │ Conjunto    │ Registros        │ Features         │
  ├─────────────┼──────────────────┼──────────────────┤
  │ TREINO      │       3,655,400 │               41 │
  │ VALIDACAO   │         522,200 │               41 │
  │ TESTE       │       1,044,400 │               41 │
  └─────────────┴──────────────────┴──────────────────┘

CARACTERISTICAS DOS DADOS:
  - Features normalizadas (StandardScaler: media=0, std=1)
  - Variaveis categoricas encodadas (Label + One-Hot)
 